# PortfolioRiskDiagnosis Review

This notebook reviews the typed diagnosis objects in `diagnosis.py` against the saved fake-dataset analysis bundle. The goal is to inspect the logic layer by layer before we depend on it in the dashboard.


## Philosophy

We are building this in object-first fashion:

- `PortfolioRiskDiagnosis` explains what is wrong overall
- `HoldingRiskContribution` explains which holdings feed those problems
- `HoldingActionRecommendation` explains what deserves action now
- `PortfolioGap` explains what the portfolio still needs before any buy ideas are shown
- `PortfolioPreferences` records the constraints that should shape future buy logic

This keeps the system explainable and easier to trust.


In [1]:
from pathlib import Path
import importlib
import json

import pandas as pd
from IPython.display import Markdown, display

import portfolio_analyzer.diagnosis as diagnosis_module
importlib.reload(diagnosis_module)

portfolio_risk_diagnosis_from_saved_artifacts = diagnosis_module.portfolio_risk_diagnosis_from_saved_artifacts

DIAGNOSIS_DIR = Path('/Users/sagnikrana/Documents/GitHub/portfolio-analyzer/data/processed/diagnosis')
OUTPUT_PATH = Path('/Users/sagnikrana/Documents/GitHub/portfolio-analyzer/data/interim/portfolio_risk_diagnosis_enriched.json')

diagnosis = portfolio_risk_diagnosis_from_saved_artifacts(DIAGNOSIS_DIR)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_rows', 200)
print('Loaded diagnosis for', diagnosis.dataset_source, 'covering', diagnosis.analysis_start, 'to', diagnosis.analysis_end)


Loaded diagnosis for Use bundled fake dataset covering 2020-08-10 to 2026-03-05


In [2]:
def build_overall_summary_df(diagnosis):
    return pd.DataFrame([
        {
            'Observed risk score': f"{diagnosis.observed_risk_score:.1f}/100",
            'Observed band': diagnosis.observed_risk_band,
            'Stated risk score': f"{diagnosis.stated_risk_score:.1f}/100",
            'Stated band': diagnosis.stated_risk_band,
            'Alignment': diagnosis.alignment,
            'Confidence': diagnosis.confidence_band,
            'Benchmark': diagnosis.benchmark_symbol,
            'Window': f"{diagnosis.analysis_start} to {diagnosis.analysis_end}",
        }
    ])


def build_top_concerns_df(diagnosis):
    return pd.DataFrame([
        {
            'Concern': item.label,
            'Base severity': item.base_severity_score,
            'External adjustment': item.external_adjustment_score,
            'Final severity': item.severity_score,
            'Band': item.severity_band,
            'Summary': item.summary,
            'Adjustment reasons': ' | '.join(item.adjustment_reasons),
            'Related tickers': ', '.join(item.related_tickers),
            'Related sectors': ', '.join(item.related_sectors),
        }
        for item in diagnosis.top_concerns
    ])


def build_top_holding_drivers_df(diagnosis):
    return pd.DataFrame([
        {
            'Ticker': item.ticker,
            'Sector': item.sector,
            'Current weight': item.current_weight,
            'Primary reason': item.primary_reason_label,
            'Primary summary': item.primary_reason_summary,
            'Secondary reasons': ' | '.join(item.secondary_reason_codes),
            'Driver confidence': item.driver_confidence_band,
            'Evidence': ' | '.join(item.evidence_summary),
        }
        for item in diagnosis.top_holding_drivers
    ])


def build_holding_contributions_df(diagnosis):
    return pd.DataFrame([
        {
            'Ticker': item.ticker,
            'Primary concern': item.primary_concern_label,
            'Overall contribution score': item.overall_contribution_score,
            'Contribution band': item.overall_contribution_band,
            'Confidence': item.contribution_confidence_band,
            'Contribution summary': item.contribution_summary,
            'Evidence': ' | '.join(item.evidence_summary),
        }
        for item in diagnosis.holding_risk_contributions
    ])


def build_holding_contribution_detail_df(diagnosis):
    rows = []
    for item in diagnosis.holding_risk_contributions:
        for detail in item.concern_contributions:
            rows.append({
                'Ticker': item.ticker,
                'Concern': detail.concern_label,
                'Contribution score': detail.contribution_score,
                'Band': detail.contribution_band,
                'Summary': detail.contribution_summary,
                'Reason codes': ' | '.join(detail.supporting_reason_codes),
                'Evidence': ' | '.join(detail.supporting_evidence),
                'Confidence': detail.confidence_band,
            })
    return pd.DataFrame(rows)


def build_holding_action_recommendations_df(diagnosis):
    return pd.DataFrame([
        {
            'Ticker': item.ticker,
            'Action': item.recommendation_label,
            'Sell / trim amount': item.position_reduction_pct,
            'Shares to sell': item.shares_to_sell,
            'Value to sell': item.value_to_sell,
            'Primary concern': item.linked_primary_concern,
            'Since-buy vs S&P 500': item.relative_performance_vs_benchmark,
            '1Y vs S&P 500': item.relative_1y_return_pct,
            '3Y vs S&P 500': item.relative_3y_return_pct,
            '5Y vs S&P 500': item.relative_5y_return_pct,
            'What changed': item.what_changed,
            'Why it matters': item.why_it_matters,
            'Why this amount': item.amount_rationale,
            'Extra modifiers': ' | '.join(item.explicit_sell_modifiers),
            'Impact preview': item.portfolio_impact_summary,
            'Guardrails': ' | '.join(item.guardrail_notes),
        }
        for item in diagnosis.holding_action_recommendations
    ])


def build_portfolio_action_impact_df(diagnosis):
    impact = diagnosis.portfolio_action_impact
    if impact is None:
        return pd.DataFrame()
    return pd.DataFrame([
        {
            'Actionable names': impact.actionable_count,
            'Tickers': ', '.join(impact.actionable_tickers),
            'Capital freed': impact.total_value_to_sell,
            'Weight reduction': impact.total_weight_reduction_pct_points,
            'Variance relief': impact.total_variance_reduction_pct_points,
            'Lag relief': impact.total_relative_drag_reduction_pct_points,
            'Largest position now': impact.current_largest_position_pct,
            'Largest position after actions': impact.projected_largest_position_pct,
            'Top 5 now': impact.current_top5_weight_pct,
            'Top 5 after actions': impact.projected_top5_weight_pct,
            'Summary': impact.impact_summary,
            'Impact bullets': ' | '.join(impact.impact_bullets),
        }
    ])


def build_portfolio_gaps_df(diagnosis):
    return pd.DataFrame([
        {
            'Gap': item.label,
            'Severity': item.severity_score,
            'Band': item.severity_band,
            'What is missing': item.what_is_missing,
            'Why this gap exists': item.why_this_gap_exists,
            'What would change': ' | '.join(item.what_would_change),
            'Linked concerns': ' | '.join(item.linked_concerns),
            'Supporting evidence': ' | '.join(item.supporting_evidence),
            'Suggested tilt': item.suggested_vehicle_tilt,
        }
        for item in diagnosis.portfolio_gaps
    ])


def build_portfolio_preferences_df(diagnosis):
    prefs = diagnosis.portfolio_preferences
    return pd.DataFrame([
        {'Preference': 'Available cash now', 'Current setting': prefs.available_cash_now},
        {'Preference': 'Available cash if actions are followed', 'Current setting': prefs.available_cash_if_actions_followed},
        {'Preference': 'Reinvest freed cash', 'Current setting': prefs.reinvestment_preference_label},
        {'Preference': 'Stated risk tolerance', 'Current setting': f"{prefs.stated_risk_score:.0f}/100 ({prefs.stated_risk_band})"},
        {'Preference': 'Suggested max size for a new add', 'Current setting': prefs.suggested_max_new_position_pct},
        {'Preference': 'ETFs allowed', 'Current setting': prefs.allow_etfs},
        {'Preference': 'Single stocks allowed', 'Current setting': prefs.allow_single_stocks},
        {'Preference': 'Default vehicle tilt', 'Current setting': prefs.vehicle_preference_label},
        {'Preference': 'Sector preferences', 'Current setting': ' | '.join(prefs.sector_preferences) if prefs.sector_preferences else 'Not set yet'},
        {'Preference': 'Inferred sector avoidances', 'Current setting': ' | '.join(prefs.inferred_sector_avoidances) if prefs.inferred_sector_avoidances else 'None inferred'},
        {'Preference': 'Constraints summary', 'Current setting': prefs.constraints_summary},
        {'Preference': 'Unresolved preferences', 'Current setting': ' | '.join(prefs.unresolved_preferences)},
        {'Preference': 'Assumption notes', 'Current setting': ' | '.join(prefs.assumption_notes)},
    ])


## 1. Overall Diagnosis Summary

Start here. This tells us whether the portfolio is behaving more aggressively or more conservatively than the user said they wanted, before we drill into holdings or actions.


In [3]:
overall_summary_df = build_overall_summary_df(diagnosis)
display(overall_summary_df)
display(Markdown('**Diagnosis summary**'))
print(diagnosis.diagnostic_summary)
print()
print('Most important concerns right now:', ', '.join(item.label for item in diagnosis.top_concerns[:3]))


,Observed risk score,Observed band,Stated risk score,Stated band,Alignment,Confidence,Benchmark,Window
0,69.2/100,Aggressive,60.0/100,Moderate,Observed portfolio risk is higher than stated risk,High,^GSPC,2020-08-10 to 2026-03-05


**Diagnosis summary**

Observed portfolio risk is 69.2/100 (Aggressive) versus a stated risk of 60.0/100 (Moderate). The portfolio currently looks higher than stated risk. The main diagnosis is driven by concentration risk, market-relative risk, over the analysis window 2020-08-10 to 2026-03-05. External evidence reinforced this read through largest holding still dominates after external review; GOOGL also carries above-market beta; macro backdrop still has restrictive rates.

Most important concerns right now: Concentration risk, Market-relative risk, Behavioral risk


## 2. Top Risk Categories (Ranked)

These are the dominant portfolio problems. `Base severity` comes from the core risk measurements, while `External adjustment` shows how much macro, filing, news, or fundamental evidence reinforced the concern.


In [4]:
top_concerns_df = build_top_concerns_df(diagnosis)
display(top_concerns_df)


,Concern,Base severity,External adjustment,Final severity,Band,Summary,Adjustment reasons,Related tickers,Related sectors
0,Concentration risk,94.1,18.0,100.0,Very high concern,"The portfolio looks concentrated in a small number of names. Largest position weight is 71.2% and top five holdings account for about 86.6% of capital. Effective holdings are only about 1.94, so d...",largest holding still dominates after external review | GOOGL also carries above-market beta | GOOGL also has risk-labeled narrative evidence | one sector dominates the portfolio along with the to...,"GOOGL, AVGO, MSFT","Communication Services, Consumer Cyclical"
1,Market-relative risk,76.1,12.0,88.1,Very high concern,"The portfolio has recently behaved more aggressively than the S&P 500. Relative volatility is about 2.03x, relative drawdown depth is about 2.09x, and market sensitivity is about 1.60x versus the ...",macro backdrop still has restrictive rates | sticky inflation can pressure richly priced risk assets | multiple main drivers still have above-market beta | external narrative evidence reinforces m...,"GOOGL, AVGO, MSFT","Communication Services, Consumer Cyclical"
2,Behavioral risk,5.3,2.5,7.8,Low concern,"Behavioral risk looks relatively contained, but it still contributes context to the diagnosis. Turnover is about 5.3% and weighted holding period is about 703 days against a target of 548 days.",several top drivers are lagging benchmark despite low trading churn | behavior should still be interpreted in light of evolving company news,,


## 3. Holding-Level Diagnosis

This section answers two questions:

1. Which holdings are currently driving the diagnosis?
2. Which specific concern does each holding feed most strongly?


In [5]:
top_holding_drivers_df = build_top_holding_drivers_df(diagnosis)
display(top_holding_drivers_df)

holding_contributions_df = build_holding_contributions_df(diagnosis)
display(holding_contributions_df)

holding_contribution_detail_df = build_holding_contribution_detail_df(diagnosis)
display(holding_contribution_detail_df)


,Ticker,Sector,Current weight,Primary reason,Primary summary,Secondary reasons,Driver confidence,Evidence
0,GOOGL,Communication Services,0.7125,Concentration pressure,GOOGL is large enough to materially influence the whole account on its own.,volatility_contributor | narrative_risk,High,Current weight: 71.2% | Variance contribution: 68.2% | 10-K filing | Investors Are Bullish on Alphabet Stock - Piling Into GOOGL Call Options With Huge Unusual Options Volume
1,AVGO,Technology,0.0089,Lagging benchmark,AVGO has underperformed the trade-matched S&P 500 since it entered the portfolio.,narrative_risk | restrictive_rates_sensitivity,High,"Excess return vs benchmark: -105.8% | 10-Q filing | Analysts Remain Bullish on Broadcom ( AVGO ) While Cathie Adds Over 143 , 000 AVGO Shares | Macro flag: rates still restrictive"
2,MSFT,Technology,0.0228,Lagging benchmark,MSFT has underperformed the trade-matched S&P 500 since it entered the portfolio.,narrative_risk | restrictive_rates_sensitivity,High,Excess return vs benchmark: -90.9% | 10-Q filing | Macro flag: rates still restrictive
3,NVDA,Technology,0.0168,Lagging benchmark,NVDA has underperformed the trade-matched S&P 500 since it entered the portfolio.,high_beta | narrative_risk,High,Excess return vs benchmark: -71.7% | Beta: 2.33 | 10-K filing
4,TSLA,Consumer Cyclical,0.0279,Lagging benchmark,TSLA has underperformed the trade-matched S&P 500 since it entered the portfolio.,narrative_risk | restrictive_rates_sensitivity,High,Excess return vs benchmark: -80.1% | 10-K filing | Macro flag: rates still restrictive


,Ticker,Primary concern,Overall contribution score,Contribution band,Confidence,Contribution summary,Evidence
0,GOOGL,Concentration risk,100.0,Very high contribution,High,"GOOGL is primarily a concentration risk driver, with secondary spillover into market-relative risk, macro sensitivity.",Current weight: 71.2% | Variance contribution: 68.2% | 10-K filing | Investors Are Bullish on Alphabet Stock - Piling Into GOOGL Call Options With Huge Unusual Options Volume
1,NVDA,Market-relative risk,86.4,Very high contribution,High,"NVDA is primarily a market-relative risk driver, with secondary spillover into company-specific risk, macro sensitivity.",Excess return vs benchmark: -71.7% | Beta: 2.33 | 10-K filing
2,TSLA,Market-relative risk,82.6,Very high contribution,High,"TSLA is primarily a market-relative risk driver, with secondary spillover into company-specific risk, macro sensitivity.",Excess return vs benchmark: -80.1% | 10-K filing | Macro flag: rates still restrictive
3,AVGO,Company-specific risk,75.4,High contribution,High,"AVGO is primarily a company-specific risk driver, with secondary spillover into market-relative risk, macro sensitivity.","Excess return vs benchmark: -105.8% | 10-Q filing | Analysts Remain Bullish on Broadcom ( AVGO ) While Cathie Adds Over 143 , 000 AVGO Shares | Macro flag: rates still restrictive"
4,MSFT,Company-specific risk,65.0,High contribution,High,"MSFT is primarily a company-specific risk driver, with secondary spillover into market-relative risk, macro sensitivity.",Excess return vs benchmark: -90.9% | 10-Q filing | Macro flag: rates still restrictive


,Ticker,Concern,Contribution score,Band,Summary,Reason codes,Evidence,Confidence
0,GOOGL,Concentration risk,100.0,Very high contribution,GOOGL mainly contributes to concentration risk through concentration pressure.,concentration_pressure,Current weight: 71.2%,High
1,GOOGL,Market-relative risk,100.0,Very high contribution,GOOGL mainly contributes to market-relative risk through volatility contributor.,volatility_contributor | narrative_risk,Variance contribution: 68.2% | 10-K filing | Investors Are Bullish on Alphabet Stock - Piling Into GOOGL Call Options With Huge Unusual Options Volume,High
2,GOOGL,Macro sensitivity,23.1,Moderate contribution,GOOGL mainly contributes to macro sensitivity through restrictive-rate sensitivity.,restrictive_rates_sensitivity,Macro flag: rates still restrictive,High
3,GOOGL,Company-specific risk,22.7,Moderate contribution,GOOGL mainly contributes to company-specific risk through narrative or event risk.,narrative_risk,10-K filing | Investors Are Bullish on Alphabet Stock - Piling Into GOOGL Call Options With Huge Unusual Options Volume,High
4,NVDA,Market-relative risk,71.2,High contribution,NVDA mainly contributes to market-relative risk through lagging benchmark.,benchmark_lag | high_beta | narrative_risk,Excess return vs benchmark: -71.7% | Beta: 2.33 | 10-K filing,High
5,NVDA,Company-specific risk,46.2,Meaningful contribution,NVDA mainly contributes to company-specific risk through lagging benchmark.,benchmark_lag | narrative_risk,Excess return vs benchmark: -71.7% | 10-K filing,High
6,NVDA,Macro sensitivity,29.7,Moderate contribution,NVDA mainly contributes to macro sensitivity through high beta.,high_beta | restrictive_rates_sensitivity,Beta: 2.33 | Macro flag: rates still restrictive,High
7,TSLA,Market-relative risk,67.2,High contribution,TSLA mainly contributes to market-relative risk through lagging benchmark.,benchmark_lag | narrative_risk | high_beta | volatility_supporting_driver,Excess return vs benchmark: -80.1% | 10-K filing | Beta: 1.92,High
8,TSLA,Company-specific risk,49.4,Meaningful contribution,TSLA mainly contributes to company-specific risk through lagging benchmark.,benchmark_lag | narrative_risk,Excess return vs benchmark: -80.1% | 10-K filing,High
9,TSLA,Macro sensitivity,27.6,Moderate contribution,TSLA mainly contributes to macro sensitivity through restrictive-rate sensitivity.,restrictive_rates_sensitivity | high_beta,Macro flag: rates still restrictive | Beta: 1.92,High


## 4. Holding Action Recommendation Review

This is the current sell / trim layer. The rule is still performance-first: a sell-style action should only appear when the holding has meaningfully lagged the trade-matched S&P 500, with diagnosis pressure and external evidence strengthening the case.


In [6]:
holding_action_recommendations_df = build_holding_action_recommendations_df(diagnosis)
display(holding_action_recommendations_df)

actionable = [item for item in diagnosis.holding_action_recommendations if item.is_actionable]
print('Actionable names on the fake dataset:')
for item in actionable:
    rel_1y = f"{item.relative_1y_return_pct:.1%}" if item.relative_1y_return_pct is not None else 'n/a'
    rel_3y = f"{item.relative_3y_return_pct:.1%}" if item.relative_3y_return_pct is not None else 'n/a'
    rel_5y = f"{item.relative_5y_return_pct:.1%}" if item.relative_5y_return_pct is not None else 'n/a'
    print(f"- {item.ticker}: {item.recommendation_label} | since-buy vs S&P 500 {item.relative_performance_vs_benchmark:.1%} | 1Y {rel_1y} | 3Y {rel_3y} | 5Y {rel_5y}")


,Ticker,Action,Sell / trim amount,Shares to sell,Value to sell,Primary concern,Since-buy vs S&P 500,1Y vs S&P 500,3Y vs S&P 500,5Y vs S&P 500,What changed,Why it matters,Why this amount,Extra modifiers,Impact preview,Guardrails
0,V,Sell all shares,1.00,32.7757,10158.49,None,-1.0520,-0.401212,-0.384710,-0.329946,V has lagged the trade-matched S&P 500 by 105.2% since the weighted-average buy date of 2023-01-04. It also underperformed in 3 of the trailing 1Y/3Y/5Y windows that were available.,This matters because the holding is still adding measurable portfolio pressure.,The current recommendation is to sell all shares because the underperformance is severe and there is not enough offsetting evidence to justify keeping even a small residual position.,,"If you make this move, the portfolio should become less concentrated, a bit steadier, and less exposed to a name that has been lagging the market.",
1,MSFT,Sell all shares,1.00,21.0475,8927.49,Company-specific risk,-0.9087,-0.188362,-0.224635,-0.064865,MSFT has lagged the trade-matched S&P 500 by 90.9% since the weighted-average buy date of 2023-10-13. It also underperformed in 3 of the trailing 1Y/3Y/5Y windows that were available.,This matters because the holding is still feeding company-specific risk through 54.1/100 diagnosis pressure.,The current recommendation is to sell all shares because the underperformance is severe and there is not enough offsetting evidence to justify keeping even a small residual position.,"**Recent company reports or news added caution** around this name, including the latest company report. | **Higher interest rates are still a headwind** for this part of the market.","If you make this move, the portfolio should become less concentrated, a bit steadier, and less exposed to a name that has been lagging the market.",
2,TSLA,Trim 35%,0.35,9.9023,3826.45,Market-relative risk,-0.8014,0.329078,0.631847,-0.134725,TSLA has lagged the trade-matched S&P 500 by 80.1% since the weighted-average buy date of 2023-01-20. It also underperformed in 1 of the trailing 1Y/3Y/5Y windows that were available.,"This matters because the holding is still feeding market-relative risk through 68.7/100 diagnosis pressure, 5.2% of tracked variance.","The current size is a 35% trim, rather than a full exit, because the system is balancing benchmark lag against the strength of the supporting evidence. Longer-horizon strength in 2 trailing window...","**Recent company reports or news added caution** around this name, including the latest company report. | **Higher interest rates are still a headwind** for this part of the market. | **This stock...","If you make this move, the portfolio should become less concentrated, a bit steadier, and less exposed to a name that has been lagging the market.",Strong trailing outperformance in multiple longer windows softened the recommendation.
3,BRK.B,Trim 25%,0.25,3.7736,1767.93,None,-0.6198,-0.445263,-0.264441,0.028479,BRK.B has lagged the trade-matched S&P 500 by 62.0% since the weighted-average buy date of 2024-04-25. It also underperformed in 2 of the trailing 1Y/3Y/5Y windows that were available.,This matters because the holding is still adding measurable portfolio pressure.,"The current size is a 25% trim, rather than a full exit, because the system is balancing benchmark lag against the strength of the supporting evidence. Longer-horizon strength in 1 trailing window...",,"If you make this move, the portfolio should become less concentrated, a bit steadier, and less exposed to a name that has been lagging the market.","Some longer-horizon strength is still present, so the recommendation is moderated rather than absolute."
4,AAPL,Trim 10%,0.10,4.0992,1091.08,None,-0.5134,0.008505,-0.096001,0.301164,AAPL has lagged the trade-matched S&P 500 by 51.3% since the weighted-average buy date of 2023-09-04. It also underperformed in 1 of the trailing 1Y/3Y/5Y windows that were available.,This matters because the holding is still adding measurable p

Actionable names on the fake dataset:
- V: Sell all shares | since-buy vs S&P 500 -105.2% | 1Y -40.1% | 3Y -38.5% | 5Y -33.0%
- MSFT: Sell all shares | since-buy vs S&P 500 -90.9% | 1Y -18.8% | 3Y -22.5% | 5Y -6.5%
- TSLA: Trim 35% | since-buy vs S&P 500 -80.1% | 1Y 32.9% | 3Y 63.2% | 5Y -13.5%
- BRK.B: Trim 25% | since-buy vs S&P 500 -62.0% | 1Y -44.5% | 3Y -26.4% | 5Y 2.8%
- AAPL: Trim 10% | since-buy vs S&P 500 -51.3% | 1Y 0.9% | 3Y -9.6% | 5Y 30.1%


## 5. Portfolio Impact Preview

This aggregates the current action set and asks: if the user actually follows these trims and sells together, what likely improves at the portfolio level?


In [7]:
portfolio_action_impact_df = build_portfolio_action_impact_df(diagnosis)
display(portfolio_action_impact_df)
if diagnosis.portfolio_action_impact is not None:
    print('Impact bullets:')
    for bullet in diagnosis.portfolio_action_impact.impact_bullets:
        print('-', bullet)


,Actionable names,Tickers,Capital freed,Weight reduction,Variance relief,Lag relief,Largest position now,Largest position after actions,Top 5 now,Top 5 after actions,Summary,Impact bullets
0,5,"V, MSFT, TSLA, BRK.B, AAPL",25771.44,0.0658,0.0618,0.06,0.7125,0.7125,0.8659,0.8589,"If you follow the current action set, the portfolio should become less concentrated, a bit steadier, and less exposed to names that have been trailing the S&P 500.","The top 5 holdings would fall from about 86.6% of the portfolio to about 85.9%. | Taken together, these moves could remove about 6.2% of recent variance contribution from the flagged names. | They..."


Impact bullets:
- The top 5 holdings would fall from about 86.6% of the portfolio to about 85.9%.
- Taken together, these moves could remove about 6.2% of recent variance contribution from the flagged names.
- They would also cut about 6.0% of weighted exposure to names that have been lagging the S&P 500.
- In total, the current action set would free up about $25,771.44 to redeploy or hold in cash.


## 6. Portfolio Gap Review

This is the new explanation-first buy-side foundation. It does **not** recommend stocks to buy. Instead, it explains what the portfolio would still be missing after the current trims and sells, and why any future add should be solving one of these specific problems.


In [8]:
portfolio_gaps_df = build_portfolio_gaps_df(diagnosis)
display(portfolio_gaps_df)

print('Plain-English readout of the fake-dataset gaps:')
for gap in diagnosis.portfolio_gaps:
    print()
    print(f'- {gap.label} ({gap.severity_score:.1f}/100)')
    print('  What is missing:', gap.what_is_missing)
    print('  Why this gap exists:', gap.why_this_gap_exists)
    print('  What would change if filled well:')
    for item in gap.what_would_change:
        print('   *', item)
    print('  Suggested tilt:', gap.suggested_vehicle_tilt)


,Gap,Severity,Band,What is missing,Why this gap exists,What would change,Linked concerns,Supporting evidence,Suggested tilt
0,Needs more diversification and steadier core exposure,100.0,Very high concern,The portfolio still needs more capital in holdings that do not depend so heavily on one or two names.,"The largest position still makes up about 71.2% of the portfolio, and even after the current trims the largest position would still be about 71.2%. The top 5 holdings currently account for about 8...",Less dependence on one holding to carry the whole portfolio. | A steadier portfolio if the biggest current winner reverses. | More room to add future ideas without immediately recreating concentra...,Concentration risk | Market-relative risk,Largest position: 71.2% | Top 5 holdings: 86.6% | Projected top 5 after current actions: 85.9%,Prefer benchmark-like core holdings or diversified funds before adding more single-name risk.
1,Needs less reliance on one sector,91.3,Very high concern,The portfolio needs more balance outside its most crowded sector.,"Communication Services currently makes up about 76.1% of invested capital, which means sector-specific weakness can spread across too much of the account.",Less cluster risk from one part of the market dominating the account. | More balanced sector exposure after the current trims free up capital. | A cleaner path to future adds that do not stack ont...,Concentration risk | Sector crowding,Top sector: Communication Services | Top sector weight: 76.1% | Lead sector driver: Sector crowding,"Favor adds outside the most crowded sector, or use broader funds that naturally dilute sector concentration."
2,"Needs steadier, lower-volatility ballast",88.1,Very high concern,The portfolio still needs holdings that behave more like a steady core than a high-swing growth sleeve.,Recent portfolio volatility has been about 2.03x the S&P 500 and market sensitivity is about 1.60x.,Smaller portfolio swings during normal market pullbacks. | Less pressure to keep trimming individual laggards just to calm the portfolio down. | A more stable base for any future stock ideas added...,Market-relative risk,Relative volatility vs S&P 500: 2.03x | Relative market sensitivity vs S&P 500: 1.60x,"Favor lower-beta, benchmark-like, or more defensive holdings over additional high-swing names."
3,Needs a clear plan for freed cash,52.0,Moderate concern,The portfolio will need a clear rule for what to do with the cash created by the current action set.,"The current trims and sells would free up about $25,771.44. Without a plan, that cash can either sit idle by accident or get pushed back into the same crowded parts of the portfolio.",Future adds can be tied to a real portfolio need instead of impulse redeployment. | Cash can be split intentionally between reinvestment and dry powder if that fits the user. | Buy recommendations...,Action follow-through,"Cash freed by current actions: $25,771.44","Decide first how much cash should be redeployed, then match that budget to the most important gap rather than the most exciting ticker."


Plain-English readout of the fake-dataset gaps:

- Needs more diversification and steadier core exposure (100.0/100)
  What is missing: The portfolio still needs more capital in holdings that do not depend so heavily on one or two names.
  Why this gap exists: The largest position still makes up about 71.2% of the portfolio, and even after the current trims the largest position would still be about 71.2%. The top 5 holdings currently account for about 86.6%, falling only to about 85.9% under the current action set.
  What would change if filled well:
   * Less dependence on one holding to carry the whole portfolio.
   * A steadier portfolio if the biggest current winner reverses.
   * More room to add future ideas without immediately recreating concentration pressure.
  Suggested tilt: Prefer benchmark-like core holdings or diversified funds before adding more single-name risk.

- Needs less reliance on one sector (91.3/100)
  What is missing: The portfolio needs more balance outside i

## 7. PortfolioPreferences / Constraints Review

This is the first typed object for future buy-side constraints. It records what we already know now and what we still need from the user before real buy recommendations would be trustworthy.


In [9]:
portfolio_preferences_df = build_portfolio_preferences_df(diagnosis)
display(portfolio_preferences_df)

prefs = diagnosis.portfolio_preferences
print('Key interpretation for the fake dataset:')
print('-', prefs.constraints_summary)
print()
print('Open user decisions before buy recommendations:')
for item in prefs.unresolved_preferences:
    print('-', item)


,Preference,Current setting
0,Available cash now,2188.9
1,Available cash if actions are followed,27960.34
2,Reinvest freed cash,Not set yet — decide whether freed cash should be reinvested or partly left in cash.
3,Stated risk tolerance,60/100 (Moderate)
4,Suggested max size for a new add,0.08
5,ETFs allowed,True
6,Single stocks allowed,True
7,Default vehicle tilt,Default to a blend of diversified ETFs and selective single stocks.
8,Sector preferences,Not set yet
9,Inferred sector avoidances,Communication Services


Key interpretation for the fake dataset:
- Right now the system can assume about $27,960.34 could be available for future adds, but reinvestment rules and sector preferences are still user decisions. Until those are set, the safest default is to favor diversified adds and keep new positions below about 8%.

Open user decisions before buy recommendations:
- Should freed cash be fully reinvested, partly reinvested, or left partly in cash?
- Do you prefer ETFs, single stocks, or a blend for new capital?
- Are there any sectors you want to emphasize or avoid?
- What is the maximum size you are comfortable giving to a new position?


## 8. Supporting Evidence

These tables stay here so we can cross-check the diagnosis logic against the underlying saved evidence without falling back to raw CSV inspection.


In [10]:
supporting_metrics_df = pd.DataFrame([item.model_dump() for item in diagnosis.supporting_metrics])
holding_fundamentals_df = pd.DataFrame([item.model_dump() for item in diagnosis.holding_fundamentals])
narrative_evidence_df = pd.DataFrame([item.model_dump() for item in diagnosis.narrative_evidence])

display(supporting_metrics_df)
display(holding_fundamentals_df)
display(narrative_evidence_df)


,metric_key,group,label,raw_value,score,score_readout,meaning,bigger_picture
0,concentration::single_position_weight,Concentration,Largest position size,0.7125,100.0,Very high concern,Checks whether one holding is large enough to dominate the portfolio.,A high score means one stock can strongly drive the whole account.
1,concentration::top_5_weight,Concentration,Top 5 holdings dominance,0.8659,100.0,Very high concern,Checks whether just a few names are carrying most of the portfolio.,A high score means your results depend heavily on a small cluster of stocks.
2,concentration::effective_holdings,Concentration,True diversification,1.9353,82.4,Very high concern,Looks past ticker count and asks how many holdings really matter after concentration is considered.,A high score means the portfolio is less diversified than it appears at first glance.
3,behavior::turnover,Behavior,Trading churn,0.0530,10.6,Low concern,Measures how much of the portfolio you rotate through selling over time.,A high score means your style is more active and decision-heavy than buy-and-hold.
4,behavior::short_holding_period,Behavior,How long your dollars stay invested,703.0000,0.0,Low concern,Measures whether larger investments are being held long enough to look like long-term investing.,A high score means bigger dollars are being cycled out faster than a long-term profile would suggest.
5,market::relative_volatility_to_benchmark,Market,Volatility vs S&P 500,2.0307,100.0,Very high concern,Checks whether your portfolio swings around more than the S&P 500 over comparable periods.,A high score means your ride has been rougher than the market's ride.
6,market::relative_drawdown_to_benchmark,Market,Downside depth vs S&P 500,2.0884,100.0,Very high concern,Checks whether your portfolio's bad stretches have been deeper than the S&P 500's.,A high score means your losses in rough periods have been worse than a simple market portfolio.
7,market::relative_downside_capture_to_benchmark,Market,Bad-day behavior vs S&P 500,1.0009,0.6,Low concern,Checks how your portfolio tends to behave on market down days compared with the S&P 500.,A high score means your portfolio tends to lose more than the market when the market is already under stress.
8,market::relative_market_sensitivity_to_benchmark,Market,Market sensitivity vs S&P 500,1.6033,80.4,Very high concern,Checks how strongly your portfolio tends to move when the S&P 500 moves.,A high score means your portfolio has been amplifying broad market moves rather than moving in line with them.
9,market::equity_exposure,Market,How fully invested you are,0.9944,99.4,Very high concern,Checks how much of your account is currently in the market rather than sitting in cash.,A high score means more of your account is directly exposed to market gains and losses right now.


,ticker,company_name,sector,industry,market_cap,beta,revenue,net_income,cash_and_equivalents,total_assets,total_liabilities,stockholders_equity,operating_cash_flow,latest_filed_date,signals
0,GOOGL,Alphabet Inc.,Communication Services,Internet Content & Information,3.837652e+12,1.128,4.028360e+11,1.321700e+11,3.070800e+10,5.952810e+11,5.122800e+10,4.152650e+11,1.647130e+11,2026-02-05,"[latest net income positive, operating cash flow positive, balance sheet coverage looks reasonable]"
1,V,Visa Inc.,Financial Services,Financial - Credit Services,5.868182e+11,0.799,2.184600e+10,5.853000e+09,1.475600e+10,9.681400e+10,1.900000e+09,2.643700e+10,6.780000e+09,2026-01-30,"[latest net income positive, operating cash flow positive, balance sheet coverage looks reasonable]"
2,TSLA,"Tesla, Inc.",Consumer Cyclical,Auto - Manufacturers,1.309411e+12,1.915,9.482700e+10,3.794000e+09,1.651300e+10,1.378060e+11,7.900000e+07,8.213700e+10,1.474700e+10,2026-01-29,"[above-market beta, latest net income positive, operating cash flow positive, balance sheet coverage looks reasonable]"
3,MSFT,Microsoft Corporation,Technology,Software - Infrastructure,2.753943e+12,1.107,2.681900e+10,3.845800e+10,2.429600e+10,6.653020e+11,-1.290100e+10,3.908750e+11,3.575800e+10,2026-01-28,"[latest net income positive, operating cash flow positive, balance sheet coverage looks reasonable]"
4,AVGO,Broadcom Inc.,Technology,Semiconductors,1.761620e+12,1.253,4.463000e+09,5.895000e+09,1.417400e+10,1.699030e+11,6.730000e+08,2.494100e+10,8.260000e+09,2026-03-11,"[above-market beta, latest net income positive, operating cash flow positive, balance sheet coverage looks reasonable]"
5,NVDA,NVIDIA Corporation,Technology,Semiconductors,4.584652e+12,2.335,2.159380e+11,1.200670e+11,1.060500e+10,2.068030e+11,3.740000e+08,1.572930e+11,1.027180e+11,2026-02-25,"[above-market beta, latest net income positive, operating cash flow positive, balance sheet coverage looks reasonable]"
6,AAPL,Apple Inc.,Technology,Consumer Electronics,3.828515e+12,1.109,2.156390e+11,4.209700e+10,4.531700e+10,3.792970e+11,1.198770e+11,8.819000e+10,5.392500e+10,2026-01-30,"[latest net income positive, operating cash flow positive, balance sheet coverage looks reasonable]"
7,VOO,Vanguard S&P 500 ETF,Financial Services,Asset Management,1.419877e+12,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,[]
8,BRK.B,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,[]
9,META,"Meta Platforms, Inc.",Communication Services,Internet Content & Information,1.587873e+12,1.309,1.372700e+10,6.045800e+10,3.587300e+10,3.660210e+11,2.210000e+09,2.172430e+11,1.158000e+11,2026-01-29,"[above-market beta, latest net income positive, operating cash flow positive, balance sheet coverage looks reasonable]"


,ticker,source_type,document_date,title,url,snippet
0,GOOGL,sec_filing,2026-02-05,10-K filing,https://www.sec.gov/Archives/edgar/data/1652044/000165204426000018/goog-20251231.htm,Risk Factors 9 Item&#160;1B. Unresolved Staff Comments 23 Item&#160;1C. Cybersecurity 23 Item&#160;2. Properties 24
1,GOOGL,news_article,None,Investors Are Bullish on Alphabet Stock - Piling Into GOOGL Call Options With Huge Unusual Options Volume,https://finance.yahoo.com/news/investors-bullish-alphabet-stock-piling-183002460.html,"Investors are trading Alphabet Inc ( GOOGL ) call options in heavy volume today, showing, for the most part, they are very bullish on Alphabet. GOOGL is down over 11.5% from its pre-earnings relea..."
2,V,sec_filing,2026-01-30,10-Q filing,https://www.sec.gov/Archives/edgar/data/1403161/000140316126000045/v-20251231.htm,Risk Factors 32 Item&#160;2. Unregistered Sales of Equity Securities and Use of Proceeds 32 Item&#160;3. Defaults Upon Senior Securities 32 Item&#160;4. Mine Safety Disclosures 32 Item&#160;5. Oth...
3,TSLA,sec_filing,2026-01-29,10-K filing,https://www.sec.gov/Archives/edgar/data/1318605/000162828026003952/tsla-20251231.htm,Item 1A. Risk Factors 12
4,MSFT,sec_filing,2026-01-28,10-Q filing,https://www.sec.gov/Archives/edgar/data/789019/000119312526027207/msft-20251231.htm,Item 1A. Risk Factors 47 &#160; &#160; &#160; &#160;
5,AVGO,sec_filing,2026-03-11,10-Q filing,https://www.sec.gov/Archives/edgar/data/1730168/000173016826000016/avgo-20260201.htm,Risk Factors 29 Item&#160;2. Unregistered Sales of Equity Securities and Use of Proceeds 47 Item&#160;3. Defaults Upon Senior Securities 47
6,AVGO,news_article,None,"Analysts Remain Bullish on Broadcom ( AVGO ) While Cathie Adds Over 143 , 000 AVGO Shares",https://finance.yahoo.com/news/analysts-remain-bullish-broadcom-avgo-180856909.html,"Broadcom Inc. (NASDAQ: AVGO ) is one of the 15 Best S&P 500 Stocks to Look For in 2026 . On January 14, Cathie Wood loaded in on Broadcom Inc. (NASDAQ:AVGO) stock, adding over 143,000 AVGO shares ..."
7,NVDA,sec_filing,2026-02-25,10-K filing,https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm,Item 1A. Risk Factors &#160; 12
8,AAPL,sec_filing,2026-01-30,10-Q filing,https://www.sec.gov/Archives/edgar/data/320193/000032019326000006/aapl-20251227.htm,Item 1A. Risk Factors 20
9,AAPL,news_article,None,Apple ( AAPL ) Postpones Its Smart Home Display Release,https://www.insidermonkey.com/blog/apple-aapl-postpones-its-smart-home-display-release-1716540/,"Apple Inc. (NASDAQ: AAPL ) earns a place on our list of the 13 unrivaled stocks of the next 10 years , though recent developments suggest the company’s artificial intelligence rollout is still evo..."


## 9. Confidence and Completeness Flags

The system should stay honest about what it knows well and what still needs more user input or richer evidence.


In [11]:
coverage_df = pd.DataFrame([diagnosis.data_coverage.model_dump()])
display(coverage_df)
print('Evidence gaps:')
for gap in diagnosis.evidence_gaps:
    print('-', gap)


,risk_metrics_available,open_positions_available,sector_allocation_available,volatility_drivers_available,performance_attribution_available,filing_text_available,news_text_available,company_facts_available,company_profiles_available,macro_series_available
0,True,True,True,True,True,True,True,True,True,True


Evidence gaps:
- User goals and constraints beyond the stated risk score are not yet modeled in the diagnosis object.
- Fundamental snapshots are based on broad canonical metrics and still need tighter metric selection for production use.


## Save Current Enriched Diagnosis

In [12]:
OUTPUT_PATH.write_text(json.dumps(diagnosis.model_dump(), indent=2) + '\n', encoding='utf-8')
print(f'Saved enriched diagnosis to {OUTPUT_PATH}')


Saved enriched diagnosis to /Users/sagnikrana/Documents/GitHub/portfolio-analyzer/data/interim/portfolio_risk_diagnosis_enriched.json
